### V0.3
* 1) age      :int64	    顧客の年齢。 
* 2) job      :object	職業（admin, blue-collar, technician など）。　
* 3) marital  :object	既婚・未婚などの婚姻状況。　
* 4) education:object	学歴（primary, secondary, tertiary など）
                    ※primary: 初等教育（小学校など）
                     secondary: 中等教育（中学校・高校など）
                     tertiary: 高等教育（大学・大学院など）
                     unknown: 不明
* 5) default  :object	債務不履行があるかどうか（yes / no）　過去にローンやクレジットカードの支払いが滞ったことがあるか
* 6) balance  :int64	    年間の平均残高。
* 7) housing  :object	住宅ローンがあるかどうか（yes / no）。
* 8) loan     :object	個人ローンがあるかどうか（yes / no）
* 9) contact  :object	連絡手段（cellular（スマホ）, telephone （固定）など）。
* 10) day      :int64	    最終接触日の「日」。
* 11) month    :object	最終接触日の「月」。
* ※　duration削除 
* 13) campaign :int64     このキャンペーン（勧誘）期間中の接触回数。
* 14) pdays    :int64	    前回のキャンペーン（勧誘）接触から経過した日数（-1は未接触）。
* 15) previous :int64	    このキャンペーン（勧誘）以前の接触回数。
* 16) poutcome :object	前回のキャンペーン（勧誘）の成果（success(契約してくれた), failure(断られた) など）。

# ライブラリ

In [57]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

# 学習モデル作成

In [58]:
df = pd.read_pickle('data.pkl')
df

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,pdays,previous,poutcome,y
0,0,42,technician,married,secondary,no,7,no,no,cellular,25,aug,3,-1,0,unknown,0
1,1,38,blue-collar,married,secondary,no,514,no,no,unknown,18,jun,1,-1,0,unknown,0
2,2,36,blue-collar,married,secondary,no,602,yes,no,unknown,14,may,2,-1,0,unknown,0
3,3,27,student,single,secondary,no,34,yes,no,unknown,28,may,2,-1,0,unknown,0
4,4,26,technician,married,secondary,no,889,yes,no,cellular,3,feb,1,-1,0,unknown,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
749995,749995,29,services,single,secondary,no,1282,no,yes,unknown,4,jul,2,-1,0,unknown,1
749996,749996,69,retired,divorced,tertiary,no,631,no,no,cellular,19,aug,1,-1,0,unknown,0
749997,749997,50,blue-collar,married,secondary,no,217,yes,no,cellular,17,apr,1,-1,0,unknown,0
749998,749998,32,technician,married,secondary,no,-274,no,no,cellular,26,aug,6,-1,0,unknown,0


In [59]:
y = df["y"]
x = df.drop(columns=["y", "id"])
x = pd.get_dummies(x, drop_first=True)


In [60]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42)

In [61]:
train_data = lgb.Dataset(x_train,label=y_train)
params = {
    'objective': 'binary',
    'boosting_type': 'gbdt',
    'metric': 'binary_logloss',
    'num_leaves': 31,
    'learning_rate': 0.05    
}

In [62]:
model = lgb.train(params, train_data,num_boost_round=100)

[LightGBM] [Info] Number of positive: 72283, number of negative: 527717
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010528 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 771
[LightGBM] [Info] Number of data points in the train set: 600000, number of used features: 41
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.120472 -> initscore=-1.987971
[LightGBM] [Info] Start training from score -1.987971


In [63]:
y_pred_proba = model.predict(x_test)
y_pred_proba

array([0.10838752, 0.68196241, 0.05366181, ..., 0.11384604, 0.12358247,
       0.33579088], shape=(150000,))

In [64]:
auc = roc_auc_score(y_test, y_pred_proba)
print("roc-auc : ",auc)

roc-auc :  0.841403093714189


# テスト、提出

In [ ]:
df_test = pd.read_pickle('test.pkl')
df_test

In [ ]:
submit_x = df_test.drop(columns=["id"])
submit_x = pd.get_dummies(submit_x, drop_first=True)
submit_x = submit_x.reindex(columns=x.columns, fill_value=0)

In [ ]:
submit_x

In [ ]:
test_pred = model.predict(submit_x)
test_pred

In [ ]:
submission = pd.DataFrame({
    'id': df_test['id'],
    'y': test_pred
})
submission.to_csv('submission.csv', index=False)